# ShadowLab - Private AI Laboratory
Run this notebook sequentially to deploy the dual-model AI inference pipeline with Cloudflare Tunnel.

In [ ]:
%%bash
echo "Phase 1: Installing System Dependencies..."
apt-get update -qq && apt-get install -y -qq \
  build-essential cmake git wget curl jq \
  libcurl4-openssl-dev \
  python3-pip python3-aiohttp python3-requests \
  2>/dev/null
nvcc --version
nvidia-smi

In [ ]:
%%bash
echo "Phase 2: Compiling llama.cpp with CUDA..."
rm -rf /opt/llama.cpp
git clone --depth 1 https://github.com/ggerganov/llama.cpp /opt/llama.cpp
cd /opt/llama.cpp

cmake -B build \
  -DGGML_CUDA=ON \
  -DLLAMA_CURL=ON \
  -DCMAKE_CUDA_ARCHITECTURES="75" \
  -DCMAKE_BUILD_TYPE=Release

cmake --build build --config Release -j$(nproc) --target llama-server

./build/bin/llama-server --version

In [ ]:
%%bash
echo "Phase 3: Downloading Models..."
MODEL_DIR="/content/models"
mkdir -p "$MODEL_DIR"

DEEPHAT_REPO="mradermacher/DeepHat-V1-7B-GGUF"
DEEPHAT_FILE="DeepHat-V1-7B.Q4_K_M.gguf"
if [ ! -f "$MODEL_DIR/$DEEPHAT_FILE" ]; then
  wget -q --show-progress -O "$MODEL_DIR/$DEEPHAT_FILE" "https://huggingface.co/${DEEPHAT_REPO}/resolve/main/${DEEPHAT_FILE}"
fi

QWEN_REPO="Qwen/Qwen2.5-VL-7B-Instruct-GGUF"
QWEN_FILE="qwen2.5-vl-7b-instruct-q4_k_m.gguf"
MMPROJ_FILE="mmproj-f16.gguf"
if [ ! -f "$MODEL_DIR/$QWEN_FILE" ]; then
  wget -q --show-progress -O "$MODEL_DIR/$QWEN_FILE" "https://huggingface.co/${QWEN_REPO}/resolve/main/${QWEN_FILE}"
fi
if [ ! -f "$MODEL_DIR/$MMPROJ_FILE" ]; then
  wget -q --show-progress -O "$MODEL_DIR/$MMPROJ_FILE" "https://huggingface.co/bartowski/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/${MMPROJ_FILE}"
fi
echo "Models downloaded to $MODEL_DIR"
df -h /

In [ ]:
%%bash
echo "Phase 4: Launching Orchestrator..."
# Assuming the orchestrator code is copied to /opt/orchestrator via git or upload
cd /content/colab_engine/orchestrator
pip install -q -r requirements.txt

pkill -f orchestrator.py || true
pkill -f llama-server || true

nohup python3 orchestrator.py \
  --listen-host 0.0.0.0 \
  --listen-port 8080 \
  --backend-port 8081 \
  --llama-server /opt/llama.cpp/build/bin/llama-server \
  --model-dir /content/models \
  > /var/log/orchestrator.log 2>&1 &

ORCH_PID=$!
echo "Orchestrator PID: $ORCH_PID"

for i in $(seq 1 20); do
  curl -sf http://localhost:8080/health && break || sleep 1
done
echo "Orchestrator Ready."

In [ ]:
%%bash
echo "Phase 5: Cloudflare Tunnel..."
wget -q -O /usr/local/bin/cloudflared "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
chmod +x /usr/local/bin/cloudflared

pkill -f cloudflared || true

cloudflared tunnel --url http://localhost:8080 \
  --no-autoupdate \
  --logfile /var/log/cloudflared.log \
  2>&1 &

TUNNEL_PID=$!
echo "Cloudflared PID: $TUNNEL_PID"

TUNNEL_URL=""
for i in $(seq 1 30); do
  TUNNEL_URL=$(grep -oP 'https://[a-z0-9-]+\.trycloudflare\.com' /var/log/cloudflared.log | head -1)
  if [ -n "$TUNNEL_URL" ]; then
    break
  fi
  sleep 2
done

if [ -z "$TUNNEL_URL" ]; then
  echo "FATAL: Could not extract tunnel URL"
  exit 1
fi

echo ""
echo "============================================================"
echo "  PUBLIC ENDPOINT: $TUNNEL_URL"
echo "============================================================"
echo ""
echo "$TUNNEL_URL" > /content/tunnel_url.txt

In [ ]:
# Phase 6: Keepalive and Monitoring
import time, requests, subprocess
from IPython.display import display, Javascript

display(Javascript('''
  setInterval(() => { google.colab.kernel.invokeFunction("keepalive", [], {}); }, 60000);
'''))

HEALTH_URL = "http://localhost:8080/health"
CHECK_INTERVAL = 120

print("🟢 Monitoring started. Keep this cell running.")
while True:
    try:
        r = requests.get(HEALTH_URL, timeout=10)
        gpu = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total,temperature.gpu", "--format=csv,noheader,nounits"],
            text=True
        ).strip()
        mem_used, mem_total, temp = gpu.split(", ")
        status = "🟢" if r.status_code == 200 else "🟡"
        print(f"{status} [{time.strftime('%H:%M:%S')}] API: {r.status_code} | GPU: {mem_used}/{mem_total} MB | Temp: {temp}°C")
    except Exception as e:
        print(f"🔴 [{time.strftime('%H:%M:%S')}] Health check failed: {e}")
    time.sleep(CHECK_INTERVAL)
